In [16]:
# Install necessary libraries. We are installing scikit-learn==1.4.2 to ensure compatibility with scikeras 0.13.0
!pip install scikeras tensorflow scikit-learn==1.4.2

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier

### Load and Prepare Data
We'll load the Iris dataset, which is a classic dataset for classification tasks.

In [18]:
data = load_iris()
X = data.data
y = data.target

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize the features (important for ANNs)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (120, 4)
y_train shape: (120,)
X_test shape: (30, 4)
y_test shape: (30,)


### Define the Keras Model
We'll create a function to build our Keras model, allowing us to pass hyperparameters like optimizer and number of neurons.

In [30]:
from tensorflow.keras.layers import Input

def build_model(optimizer='adam', neurons=8):
    model = Sequential()
    # Input layer and first hidden layer (using Input(shape) for best practice)
    model.add(Input(shape=(X_train.shape[1],)))
    model.add(Dense(neurons, activation='relu'))
    # Second hidden layer
    model.add(Dense(neurons, activation='relu'))
    # Output layer (3 classes for Iris, using softmax for multi-class classification)
    model.add(Dense(len(np.unique(y)), activation='softmax'))

    # Compile the model
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy', # Use sparse_categorical_crossentropy for integer labels
        metrics=['accuracy']
    )
    return model

### Train and Evaluate the Keras Model
We will directly train the Keras model with a set of default parameters and then evaluate its performance on the test set.

In [28]:
# Create the KerasClassifier wrapper for scikit-learn compatibility
model_instance = KerasClassifier(model=build_model,
                                 optimizer='adam',
                                 neurons=16,
                                 batch_size=16,
                                 epochs=100,
                                 verbose=0)

# Train the model
print("Starting model training...")
model_instance.fit(X_train, y_train)
print("Model training finished.")

Starting model training...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model training finished.


### Evaluate Results
Let's examine the performance of the trained model.

In [29]:
# Make predictions on the test set
y_pred_proba = model_instance.predict_proba(X_test)
y_pred = np.argmax(y_pred_proba, axis=1)

# Evaluate the trained model on the test set
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy with Trained Model: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test Accuracy with Trained Model: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30


Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
